In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    dane = message.value
    
    if dane['amount'] > 3000:
        print(f"ALERT: {dane['tx_id']} | {dane['amount']} PLN | {dane['store']} | {dane['category']}")

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='grupa_zadanie3', 
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    kwota = tx['amount']
    
    if kwota > 3000:
        tx['risk_level'] = "HIGH"
    elif kwota > 1000:
        tx['risk_level'] = "MEDIUM"
    else:
        tx['risk_level'] = "LOW"
        
    print(tx)

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='grupa_sklepy',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
licznik = 0

for message in consumer:
    tx = message.value
    sklep = tx['store']
    
    # zliczanie
    store_counts[sklep] += 1
    
    if sklep not in total_amount:
        total_amount[sklep] = 0.0
    total_amount[sklep] += tx['amount']
    
    licznik += 1
    
    # co 10 wiadomosci
    if licznik % 10 == 0:
        print("Sklep | Liczba | Suma | Srednia")
        for s in store_counts:
            srednia = total_amount[s] / store_counts[s]
            print(f"{s} | {store_counts[s]} | {round(total_amount[s], 2)} | {round(srednia, 2)}")
        print("")

Writing consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='grupa_statystyki',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = {}
i = 0

for message in consumer:
    tx = message.value
    kat = tx['category']
    kwota = tx['amount']
    
    if kat not in stats:
        stats[kat] = {'ilosc': 0, 'suma': 0.0, 'min': kwota, 'max': kwota}
        
    stats[kat]['ilosc'] += 1
    stats[kat]['suma'] += kwota
    
    if kwota < stats[kat]['min']:
        stats[kat]['min'] = kwota
    if kwota > stats[kat]['max']:
        stats[kat]['max'] = kwota
        
    i += 1
    if i % 10 == 0:
        print("Statystyki kategorii:")
        for k, v in stats.items():
            print(f"{k} -> ilosc: {v['ilosc']}, suma: {round(v['suma'], 2)}, min: {v['min']}, max: {v['max']}")
        print("===")

Writing consumer_stats.py
